# Migrating Existing Checkpoints: `CEModel` + the Legacy Shim

The `general` CE package keeps every checkpoint produced by the paper's
original `CenterEmbeddingResNetv4`/`v5` classes loadable **with no file
conversion**. This notebook demonstrates the wrapper end to end:

1. Wrap the frozen legacy model classes as `CEModel` encoders
   (`legacy_v4_center` / `legacy_v4_context`, or the `v5` equivalents).
2. Load a legacy flat `state_dict` (keys prefixed `0.*`/`1.*`, exactly what
   `torch.save(nn.Sequential(center, context).state_dict(), path)` produces)
   through `CEModel.from_legacy_checkpoint`, which remaps those prefixes.
3. Run the *same* batch through the untouched legacy classes and through the
   wrapped `CEModel`, and confirm the logits match to floating-point
   tolerance.

Real training checkpoints (`swmodel_{version}_E{embedding}_B{batch}_topk_{score}.pth`)
live on the IBEX cluster at `/ibex/scratch/alveardf/slurm_outputs/weights/`
(CLAUDE.md §8) and are not part of this repo. The checkpoint-loading cell
below is wrapped so that its absence doesn't stop the notebook: if no local
`.pth` is found, we fall back to a **freshly-initialized** legacy model pair
(still going through the exact same `nn.Sequential` state-dict round trip) to
demonstrate the wrapper's shape/key compatibility without a real trained
checkpoint.

In [1]:
import glob
import os
import tempfile

import torch
from torch import nn

from conditional_embedding_model.models.center_embedding_resnet import (
    CenterEmbeddingResNetv4, ContextEmbeddingResNetv4,
    CenterEmbeddingResNetv5, ContextEmbeddingResNetv5,
)
from conditional_embedding_model.general import (
    CEModel, CEModelConfig, EncoderConfig, ScorerConfig, parse_legacy_filename,
)

FEATURE_STRUCTURE = {
    "features_structured": {"map": 8100, "object_footprint": 8, "grasping_approach": 600},
    "features_shape": {"map": (90, 90), "object_footprint": (4, 2), "grasping_approach": (150, 4)},
}


pybullet build time: Jan 29 2025 23:16:28


## 1. Legacy checkpoint format

A saved checkpoint is just `nn.Sequential(center_model, context_model).state_dict()`
— no optimizer, scheduler, or config. Keys are prefixed by Sequential index:
**`0.*`** belongs to the center model, **`1.*`** to the context model (e.g.
`0.embedder.0.weight`, `1.context_embedding_gru.weight_ih_l0`). The config
implied by a checkpoint (model version, `embedding_dim`) has to come from the
filename convention, since it isn't stored in the file itself:
`parse_legacy_filename` codifies that.

In [2]:
example = parse_legacy_filename("swmodel_v4_E83_B32_topk_0.912.pth")
example


{'model_version': 'v4',
 'embedding_size': 83,
 'batch_size': 32,
 'hit_at_1': 0.912}

## 2. Locating a real checkpoint (optional, non-fatal)

We look for any `.pth` matching the legacy naming convention under a few
plausible local locations, plus the IBEX path from CLAUDE.md §8 (which will
not exist off-cluster). If nothing is found, we print a clear message and
continue with a freshly-initialized model pair instead — the rest of the
notebook runs identically either way, since `from_legacy_checkpoint` doesn't
care whether the weights are trained or random.

In [3]:
_SEARCH_DIRS = [
    "/ibex/scratch/alveardf/slurm_outputs/weights",
    os.path.join(os.path.dirname(os.getcwd()), "config", "weights"),
    os.path.join(os.getcwd(), "..", "config", "weights"),
]

found_checkpoint = None
for d in _SEARCH_DIRS:
    try:
        matches = sorted(glob.glob(os.path.join(d, "swmodel_*.pth")))
    except OSError:
        matches = []
    if matches:
        found_checkpoint = matches[0]
        break

if found_checkpoint is not None:
    print(f"Found local checkpoint: {found_checkpoint}")
else:
    print(
        "No local checkpoint matching 'swmodel_*.pth' found under:\n  "
        + "\n  ".join(_SEARCH_DIRS)
        + "\nThis is expected off the IBEX cluster (CLAUDE.md \u00a78) -- "
        "falling back to a freshly-initialized legacy model pair to "
        "demonstrate the wrapper's shape/key compatibility instead."
    )


No local checkpoint matching 'swmodel_*.pth' found under:
  /ibex/scratch/alveardf/slurm_outputs/weights
  /home/davidthinkpad/Documents/projects/P-CooperativeGrasping/conditional_embedding_model/config/weights
  /home/davidthinkpad/Documents/projects/P-CooperativeGrasping/conditional_embedding_model/notebooks/../config/weights
This is expected off the IBEX cluster (CLAUDE.md §8) -- falling back to a freshly-initialized legacy model pair to demonstrate the wrapper's shape/key compatibility instead.


## 3. Building (or synthesizing) a legacy checkpoint file

If a real checkpoint was found, we use it directly along with the config
parsed from its filename. Otherwise we build a fresh legacy `v4` center/context
pair, save it exactly the way the original training code does
(`nn.Sequential(center, context).state_dict()` via `torch.save`), and treat
that as our "checkpoint" — same file format, untrained weights.

In [4]:
if found_checkpoint is not None:
    parsed = parse_legacy_filename(found_checkpoint)
    model_version = parsed["model_version"]
    embedding_dim = parsed["embedding_size"]
    checkpoint_path = found_checkpoint
    _tmp_path = None
else:
    model_version = "v4"
    embedding_dim = 64
    torch.manual_seed(1234)
    legacy_center = CenterEmbeddingResNetv4(
        input_dim=4, embedding_dim=embedding_dim, feature_structure=FEATURE_STRUCTURE, dropout=0.1)
    legacy_context = ContextEmbeddingResNetv4(
        input_dim=4, embedding_dim=embedding_dim, feature_structure=FEATURE_STRUCTURE, dropout=0.1)
    legacy_seq = nn.Sequential(legacy_center, legacy_context)

    _tmp = tempfile.NamedTemporaryFile(suffix=".pth", delete=False)
    _tmp_path = _tmp.name
    _tmp.close()
    torch.save(legacy_seq.state_dict(), _tmp_path)
    checkpoint_path = _tmp_path

print(f"model_version={model_version!r}, embedding_dim={embedding_dim}, checkpoint_path={checkpoint_path}")


/home/davidthinkpad/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/davidthinkpad/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


model_version='v4', embedding_dim=64, checkpoint_path=/tmp/tmpzl6xqjxv.pth


## 4. Wrapping via `CEModel.from_legacy_checkpoint`

The config below names `legacy_v4_center`/`legacy_v4_context` (or `v5`
equivalents) as the encoders and `cosine_temperature` as the scorer — exactly
reproducing `prediction_temperature` from the original `trainer.py`.
`from_legacy_checkpoint` builds this `CEModel`, remaps the flat `0.*`/`1.*`
keys to `center_encoder.inner.*`/`context_encoder.inner.*`, and loads them
with `strict=True` (the scorer is parameter-free, so nothing is left over).

In [5]:
center_name = f"legacy_{model_version}_center"
context_name = f"legacy_{model_version}_context"

config = CEModelConfig(
    embedding_dim=embedding_dim,
    center_encoder=EncoderConfig(name=center_name, params=dict(
        input_dim=4, embedding_dim=embedding_dim, feature_structure=FEATURE_STRUCTURE,
        dropout=0.1, eval_safe_transform=False,
    )),
    context_encoder=EncoderConfig(name=context_name, params=dict(
        input_dim=4, embedding_dim=embedding_dim, feature_structure=FEATURE_STRUCTURE,
        dropout=0.1, eval_safe_transform=False,
    )),
    scorer=ScorerConfig(name="cosine_temperature", params={"temperature": 0.07}),
)

try:
    cemodel = CEModel.from_legacy_checkpoint(checkpoint_path, config, strict=True)
    print("Loaded checkpoint into CEModel successfully; state_dict keys match strictly.")
except Exception as e:
    print(f"Checkpoint load failed ({type(e).__name__}: {e}); this notebook cannot "
          "continue the comparison without a loadable checkpoint.")
    raise


Loaded checkpoint into CEModel successfully; state_dict keys match strictly.


## 5. Comparing legacy vs. wrapped logits on a fixed batch

To compare fairly we need the *same underlying weights* on both sides. When we
had a real checkpoint we only have the `CEModel` side loaded from it; to get an
equivalent legacy `nn.Sequential` for comparison we build one and load the same
file into it directly (both sides read from the same `checkpoint_path`, so
this is an honest apples-to-apples check either way).

**Why the seed resets matter:** the legacy `forward()` methods apply
`RandomHorizontalFlip`/`RandomAffine`/`ColorJitter` to the map image with no
`if self.training:` guard (a known bug, see CLAUDE.md §7) — augmentation runs
even in `.eval()` mode and consumes RNG state on every call. `eval_safe_transform=False`
above means the wrapped encoder reproduces this behavior bit-for-bit rather than
routing around it, so for the two forward passes to land on identical random
draws we must reset `torch.manual_seed` immediately before *each* one, exactly
as `tests/test_legacy_equivalence.py` does.

In [6]:
if model_version == "v4":
    center_cls, context_cls = CenterEmbeddingResNetv4, ContextEmbeddingResNetv4
else:
    center_cls, context_cls = CenterEmbeddingResNetv5, ContextEmbeddingResNetv5

legacy_center = center_cls(input_dim=4, embedding_dim=embedding_dim, feature_structure=FEATURE_STRUCTURE, dropout=0.1)
legacy_context = context_cls(input_dim=4, embedding_dim=embedding_dim, feature_structure=FEATURE_STRUCTURE, dropout=0.1)
legacy_seq = nn.Sequential(legacy_center, legacy_context)
legacy_state = torch.load(checkpoint_path, map_location="cpu")
legacy_seq.load_state_dict(legacy_state)

legacy_seq.eval()
cemodel.eval()


def prediction_temperature(center, contexts_and_negatives, net_v, net_u, feature, temperature=0.07):
    v = torch.nn.functional.normalize(net_v(center, feature), dim=-1)
    u = torch.nn.functional.normalize(net_u(contexts_and_negatives, feature), dim=-1)
    pred = torch.bmm(v.unsqueeze(1), u.permute(0, 2, 1)) / temperature
    return pred.squeeze(1)


torch.manual_seed(1234)
feature = torch.rand(4, 8708)
center = torch.zeros(4, 1)
contexts = torch.randint(0, 150, (4, 10))

torch.manual_seed(0)
legacy_logits = prediction_temperature(center, contexts, legacy_seq[0], legacy_seq[1], feature, temperature=0.07)

torch.manual_seed(0)
new_logits = cemodel({"index": center, "feature": feature}, {"index": contexts, "feature": feature})

max_diff = (legacy_logits - new_logits).abs().max().item()
print("legacy logits:\n", legacy_logits)
print("CEModel logits:\n", new_logits)
print(f"max |legacy - CEModel| = {max_diff:.3e}")
assert torch.allclose(legacy_logits, new_logits, atol=1e-6), f"logits mismatch: {max_diff}"
print("PASS: legacy and CEModel logits match within atol=1e-6.")


legacy logits:
 tensor([[3.3391, 3.5335, 3.6119, 3.8156, 3.8669, 3.7338, 3.5352, 3.2378, 3.0166,
         2.7155],
        [3.3892, 3.5168, 3.6371, 3.7038, 3.7500, 3.7281, 3.6723, 3.3033, 2.9442,
         2.6374],
        [3.2787, 3.4047, 3.3788, 3.3311, 3.3451, 3.3644, 3.2232, 2.9168, 2.5893,
         2.1703],
        [3.3863, 3.3080, 3.4228, 3.4257, 3.4507, 3.3724, 3.3194, 3.1774, 2.9436,
         2.6251]], grad_fn=<SqueezeBackward1>)
CEModel logits:
 tensor([[3.3391, 3.5335, 3.6119, 3.8156, 3.8669, 3.7338, 3.5352, 3.2378, 3.0166,
         2.7155],
        [3.3892, 3.5168, 3.6371, 3.7038, 3.7500, 3.7281, 3.6723, 3.3033, 2.9442,
         2.6374],
        [3.2787, 3.4047, 3.3788, 3.3311, 3.3451, 3.3644, 3.2232, 2.9168, 2.5893,
         2.1703],
        [3.3863, 3.3080, 3.4228, 3.4257, 3.4507, 3.3724, 3.3194, 3.1774, 2.9436,
         2.6251]], grad_fn=<SqueezeBackward1>)
max |legacy - CEModel| = 0.000e+00
PASS: legacy and CEModel logits match within atol=1e-6.


## 6. Round-tripping back to a legacy-loadable state dict

`to_legacy_state_dict()` inverts the remap, so a `CEModel` trained through the
new `Trainer` still produces a file the original (untouched) notebooks
(`define_model` + `load_state_dict`) can load directly.

In [7]:
roundtrip_state = cemodel.to_legacy_state_dict()
legacy_keys = set(torch.load(checkpoint_path, map_location="cpu").keys())
assert set(roundtrip_state.keys()) == legacy_keys, "round-trip key set does not match legacy checkpoint"
mismatches = [k for k in legacy_keys if not torch.equal(roundtrip_state[k], legacy_state[k])]
assert not mismatches, f"round-trip tensor mismatch at keys: {mismatches}"
print(f"PASS: to_legacy_state_dict() round-trips exactly ({len(legacy_keys)} keys, all tensors identical).")


PASS: to_legacy_state_dict() round-trips exactly (408 keys, all tensors identical).


In [8]:
if _tmp_path is not None:
    os.remove(_tmp_path)


## Summary

- `CEModel.from_legacy_checkpoint(path, config)` loads any flat `0.*`/`1.*`
  legacy state dict into a composed `CEModel`, given a config that names the
  right `legacy_v4_*`/`legacy_v5_*` encoders and `embedding_dim` (recoverable
  from the filename via `parse_legacy_filename`, or supplied manually).
- The wrapped model reproduces the frozen legacy classes' logits exactly
  (`atol=1e-6`), including their inference-time augmentation quirk when
  `eval_safe_transform=False` — the shim is a faithful wrapper, not a
  reimplementation.
- `to_legacy_state_dict()` inverts the mapping so training through the new
  `Trainer` remains compatible with every existing legacy consumer.
- When run off the IBEX cluster, this notebook degrades gracefully: no real
  checkpoint is required for the comparison to be meaningful, since the shim's
  correctness doesn't depend on the weights being trained.